In [ ]:
import pandas as pd
from os.path import join
from bertopic import BERTopic
from topics_to_merge import topics_to_merge
from config import DATA_DIR

In [21]:
topic_model = BERTopic.load("topic_models/251219")

In [ ]:
# load cleaned toot sample per instance at max. 75 percentile
df = pd.read_csv(join(DATA_DIR, "toots_per_instance_sampled_75_percentile.csv"))

# recalculate embeddings for/and outlier reduction

In [12]:
docs = df["cleaned_content"].tolist()
embeddings = topic_model.embedding_model.embed(docs)

In [26]:
new_topics = topic_model.reduce_outliers(
    documents=docs,
    topics=topic_model.topics_,
    embeddings=embeddings,
    strategy="embeddings",
    threshold=0.015
)

In [67]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,233030,0_twitter_يا_status_じぇ,"[twitter, يا, status, じぇ, حافظ, يلعن, روحك, yo...",NaN
1,1,75723,1_music_apple_album_song,"[music, apple, album, song, bandcamp, iphone, ...",NaN
2,2,39595,2_mastodon_instance_social_fediverse,"[mastodon, instance, social, fediverse, v4, se...",NaN
3,3,41552,3_football_sport_f1_nfl,"[football, sport, f1, nfl, season, hockey, tea...",NaN
4,4,39010,4_climate_mac_usb_macos,"[climate, mac, usb, macos, ram, laptop, intel,...",NaN
...,...,...,...,...,...
357,357,1062,357_shave_wetshaving_razor_lather,"[shave, wetshaving, razor, lather, brush, sotd...",NaN
358,358,905,358_tennis_djokovic_racket_novak,"[tennis, djokovic, racket, novak, atp, wimbled...",NaN
359,359,11394,359_fuck_fucking_shit_pussy,"[fuck, fucking, shit, pussy, cock, hell, oh, h...",NaN
360,360,3447,360_98_197_76_88,"[98, 197, 76, 88, 395, cve, test, 1318, 51, 52]",NaN


In [29]:
topic_model.update_topics(docs,topics=new_topics)

2025-12-25 11:26:02,009 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [69]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,233030,0_twitter_يا_status_じぇ,"[twitter, يا, status, じぇ, حافظ, يلعن, روحك, yo...",NaN
1,1,75723,1_music_apple_album_song,"[music, apple, album, song, bandcamp, iphone, ...",NaN
2,2,39595,2_mastodon_instance_social_fediverse,"[mastodon, instance, social, fediverse, v4, se...",NaN
3,3,41552,3_football_sport_f1_nfl,"[football, sport, f1, nfl, season, hockey, tea...",NaN
4,4,39010,4_climate_mac_usb_macos,"[climate, mac, usb, macos, ram, laptop, intel,...",NaN
...,...,...,...,...,...
357,357,1062,357_shave_wetshaving_razor_lather,"[shave, wetshaving, razor, lather, brush, sotd...",NaN
358,358,905,358_tennis_djokovic_racket_novak,"[tennis, djokovic, racket, novak, atp, wimbled...",NaN
359,359,11394,359_fuck_fucking_shit_pussy,"[fuck, fucking, shit, pussy, cock, hell, oh, h...",NaN
360,360,3447,360_98_197_76_88,"[98, 197, 76, 88, 395, cve, test, 1318, 51, 52]",NaN


In [ ]:
topic_model.save(join("topic_models","251219_reduced_outliers"), serialization="safetensors", save_ctfidf=True, save_embedding_model="paraphrase-multilingual-MiniLM-L12-v2")

# merge topics

In [245]:
reduced_outliers_model = BERTopic.load(join("topic_models", "251219_reduced_outliers"))

In [246]:
reduced_outliers_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,233030,0_twitter_يا_status_じぇ,"[twitter, يا, status, じぇ, حافظ, يلعن, روحك, yo...",NaN
1,1,75723,1_music_apple_album_song,"[music, apple, album, song, bandcamp, iphone, ...",NaN
2,2,39595,2_mastodon_instance_social_fediverse,"[mastodon, instance, social, fediverse, v4, se...",NaN
3,3,41552,3_football_sport_f1_nfl,"[football, sport, f1, nfl, season, hockey, tea...",NaN
4,4,39010,4_climate_mac_usb_macos,"[climate, mac, usb, macos, ram, laptop, intel,...",NaN
...,...,...,...,...,...
357,357,1062,357_shave_wetshaving_razor_lather,"[shave, wetshaving, razor, lather, brush, sotd...",NaN
358,358,905,358_tennis_djokovic_racket_novak,"[tennis, djokovic, racket, novak, atp, wimbled...",NaN
359,359,11394,359_fuck_fucking_shit_pussy,"[fuck, fucking, shit, pussy, cock, hell, oh, h...",NaN
360,360,3447,360_98_197_76_88,"[98, 197, 76, 88, 395, cve, test, 1318, 51, 52]",NaN


## check that all topics are categorized

In [247]:
info = reduced_outliers_model.get_topic_info().set_index("Topic")
topic_labels = dict(zip(info.index, info["Name"]))

for i, group in enumerate(topics_to_merge, start=1):
    print(f"group {i}:")
    for tid in group:
        label = topic_labels.get(tid, "<UNKNOWN>")
        print(f"{label}")
    print() 

group 1:
0_twitter_يا_status_じぇ
2_mastodon_instance_social_fediverse
7_youtube_watch_twitch_video
51_blog_internet_web_post
53_musk_elon_twitter_elonmusk
75_threads_thread_activitypub_meta
92_account_accounts_instance_my
110_bluesky_bsky_invite_blue
115_facebook_meta_instagram_fb
140_reddit_irc_stopdronebl_liberachat
168_fediverse_welcome_fediverso_observer
174_friends_friend_friendship_friendica
180_post_posts_posting_here
195_fedi_fedora_fedilab_fediblock
201_pleroma_pr_shc_kanagawa_jp_clojure_instance
203_rss_feed_feeds_reader
230_instagram_account_photos_image
233_pixelfed_pixel_image_images
238_reddit_exmormon_fade_comments
246_whatsapp_signal_telegram_messenger
248_ivory_tapbots_tweetbot_app
256_hashtags_hashtag_span_tags
262_johnmastodon_funfacts_john_mastodon
308_followers_fandoms_follow_follower
342_blue_imastodon_bluey_blaue

group 2:
1_music_apple_album_song
43_podcast_podcasts_audio_episode
131_dance_dancing_ballet_dances
156_eurovision_eurovision2023_x27_esc
197_spotify_pl

In [248]:
reduced_outliers_model.merge_topics(docs, topics_to_merge)

In [252]:
reduced_outliers_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,414066,0_mastodon_to_39_twitter,"[mastodon, to, 39, twitter, com, the, it, and,...",[This is my first post on Mastodon. I have no ...
1,1,236531,1_39_to_the_and,"[39, to, the, and, you, it, of, is, in, my]",[If you know what you are doing 3/4 of the tim...
2,2,153861,2_the_in_to_and,"[the, in, to, and, of, quot, de, 2023, 39, it]",[The time now is: Australia/Sydney: 2023-11-13...
3,3,148550,3_to_the_and_39,"[to, the, and, 39, it, for, of, is, in, that]",[Glad to see more Rhapsody coverage. When I wa...
4,4,144579,4_de_in_the_der,"[de, in, the, der, quot, of, to, la, 39, die]","[Why Waterford city? Well, I&#39;m from Waterf..."
5,5,130588,5_to_the_it_39,"[to, the, it, 39, and, is, for, on, my, linux]",[Going forward my direct contact lines will be...
6,6,123786,6_to_and_the_39,"[to, and, the, 39, of, my, in, it, that, is]",[@ DarcMoughty It depends on how you define it...
7,7,121603,7_die_quot_zøt_akkoma,"[die, quot, zøt, akkoma, e5, the, natty, 39, e...",[保安局指美國制裁釀資安漏洞 擬斥1.9億換人事紀錄系統 inmediahk.net/nod...
8,8,118029,8_the_of_and_39,"[the, of, and, 39, to, it, in, is, quot, that]",[@ libreture This is nice. My problem with Goo...
9,9,109368,9_music_the_apple_com,"[music, the, apple, com, 39, to, and, of, on, ...",[Now playing on Mixfix Radio: backyardigans do...


In [237]:
reduced_outliers_model.save(join("topic_models", "251219_reduced_outliers_merged_topics"), serialization="safetensors", save_ctfidf=True, save_embedding_model="paraphrase-multilingual-MiniLM-L12-v2")

# transform docs with merged model to calculate new topics & probs per item in docs

In [13]:
reduced_outliers_merged_model = BERTopic.load(join("topic_models","251219_reduced_outliers_merged_topics"))

In [14]:
merged_topics, merged_probs = reduced_outliers_merged_model.transform(docs)

In [15]:
merged_probs

array([0.22341305, 0.4940958 , 0.606624  , ..., 0.23532455, 0.28417882,
       0.36946613], dtype=float32)

In [16]:
df["topic"] = merged_topics 
df["topic_prob"] = merged_probs.max()

In [18]:
reduced_outliers_merged_model.get_topics()

{0: [['mastodon', 0.02813834050417026],
  ['to', 0.02209875650144803],
  ['39', 0.02133875598733121],
  ['twitter', 0.021222892512136913],
  ['com', 0.020443318387746785],
  ['the', 0.01877088923349335],
  ['it', 0.016149087837199972],
  ['and', 0.015765332521955774],
  ['on', 0.015567063241212271],
  ['youtube', 0.01534117745116656]],
 1: [['39', 0.021974710192686497],
  ['to', 0.021825483126645562],
  ['the', 0.021552434677047667],
  ['and', 0.019137646103760442],
  ['you', 0.01743902373672628],
  ['it', 0.01722480509763552],
  ['of', 0.016375636352700434],
  ['is', 0.015026040556018153],
  ['in', 0.014819706231184577],
  ['my', 0.014674335422366151]],
 2: [['the', 0.02215849162674519],
  ['in', 0.017805257121377687],
  ['to', 0.01583532394831263],
  ['and', 0.015230187311926324],
  ['of', 0.015226625026370845],
  ['quot', 0.014830180202008547],
  ['de', 0.014313165788512468],
  ['2023', 0.01420120608723962],
  ['39', 0.01367891472318234],
  ['it', 0.013143919952622348]],
 3: [['to',

In [23]:
reduced_outliers_merged_model.set_topic_labels({0: "social_media_plaforms_communities", 
 1: "life_relationship",
 2: "environment_climate_weather",
 3: "software_programming",
 4: "countries_local_contexts",
 5: "hardware_devices_infrastructure",
 6: "health_mental_health_body",
 7: "misc",
 8: "books_movies_fandom",
 9: "music", 
 10: "arts_crafts_design", 
 11: "current_political_events",
 12: "science_research",
 13: "travel_mobility_transport",
 14: "food_drinks",
 15: "identity_discrimination_subculture",
 16: "economy_finance", 
 17: "cloud_networks_security_privacy",
 18: "animals_nature",
 19: "news_politics",
 20: "games_gaming",
 21: "sports",
 22: "humor_memes_play",
 23: "religion",
 24: "ai_silicon_valley",
 25: "adult_content",
})

In [24]:
reduced_outliers_merged_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,0,414066,0_mastodon_to_39_twitter,social_media_plaforms_communities,"[mastodon, to, 39, twitter, com, the, it, and,...",NaN
1,1,236531,1_39_to_the_and,life_relationship,"[39, to, the, and, you, it, of, is, in, my]",NaN
2,2,153861,2_the_in_to_and,environment_climate_weather,"[the, in, to, and, of, quot, de, 2023, 39, it]",NaN
3,3,148550,3_to_the_and_39,software_programming,"[to, the, and, 39, it, for, of, is, in, that]",NaN
4,4,144579,4_de_in_the_der,countries_local_contexts,"[de, in, the, der, quot, of, to, la, 39, die]",NaN
5,5,130588,5_to_the_it_39,hardware_devices_infrastructure,"[to, the, it, 39, and, is, for, on, my, linux]",NaN
6,6,123786,6_to_and_the_39,health_mental_health_body,"[to, and, the, 39, of, my, in, it, that, is]",NaN
7,7,121603,7_die_quot_zøt_akkoma,misc,"[die, quot, zøt, akkoma, e5, the, natty, 39, e...",NaN
8,8,118029,8_the_of_and_39,books_movies_fandom,"[the, of, and, 39, to, it, in, is, quot, that]",NaN
9,9,109368,9_music_the_apple_com,music,"[music, the, apple, com, 39, to, and, of, on, ...",NaN


# n top topics per server

In [25]:
def calculate_top_n(df, topic_model, n=10): 

    server_topic_weighted = (
        df.groupby(["instance", "topic"])["topic_prob"]
        .sum()
        .reset_index(name="weighted_prob_sum")
    )
    
    top_topics_per_server = (
        server_topic_weighted.sort_values(["instance", "weighted_prob_sum"], ascending=[True, False])
        .groupby("instance")
        .head(n)
    )
    
    topic_info_df = topic_model.get_topic_info()[["Topic", "CustomName"]]
    topic_info_dict = dict(zip(topic_info_df["Topic"], topic_info_df["CustomName"]))
    top_topics_per_server["topic_name"] = top_topics_per_server["topic"].map(topic_info_dict)
    
    server_total = top_topics_per_server.groupby("instance")["weighted_prob_sum"].transform("sum")
    top_topics_per_server["relative_share"] = top_topics_per_server["weighted_prob_sum"] / server_total
    
    
    server_topics_dict = (
        top_topics_per_server.groupby("instance")[["topic_name", "relative_share"]]
        .apply(lambda x: list(zip(x["topic_name"], x["relative_share"])))
        .to_dict()
    )

    return server_topics_dict

In [27]:
server_topics_dict = calculate_top_n(df, reduced_outliers_merged_model, n=26)

In [28]:
server_topics_dict

{'0.0n.ee': [('ai_silicon_valley', 0.19298246502876282),
  ('social_media_plaforms_communities', 0.17543861269950867),
  ('cloud_networks_security_privacy', 0.12280701845884323),
  ('misc', 0.10526315867900848),
  ('humor_memes_play', 0.08771930634975433),
  ('countries_local_contexts', 0.07017543911933899),
  ('software_programming', 0.05263157933950424),
  ('hardware_devices_infrastructure', 0.05263157933950424),
  ('food_drinks', 0.035087719559669495),
  ('health_mental_health_body', 0.017543859779834747),
  ('music', 0.017543859779834747),
  ('science_research', 0.017543859779834747),
  ('economy_finance', 0.017543859779834747),
  ('animals_nature', 0.017543859779834747),
  ('religion', 0.017543859779834747)],
 '000000420.xyz': [('social_media_plaforms_communities', 0.25),
  ('life_relationship', 0.125),
  ('software_programming', 0.125),
  ('health_mental_health_body', 0.125),
  ('travel_mobility_transport', 0.125),
  ('food_drinks', 0.125),
  ('economy_finance', 0.125)],
 '0rb.it

In [41]:
all_instances = df["instance"].unique()
all_topic_names = set(topic for topics in server_topics_dict.values() for topic, _ in topics)

rows = []
for instance in all_instances:
    row = {"instance": instance}
    topic_dict = dict(server_topics_dict.get(instance, []))
    
    for topic_name in all_topic_names:
        row[topic_name] = topic_dict.get(topic_name, 0)
    
    rows.append(row)

server_topic_df = pd.DataFrame(rows)

In [53]:
server_topic_df.columns

Index(['instance', 'software_programming', 'misc', 'science_research',
       'news_politics', 'humor_memes_play', 'sports',
       'countries_local_contexts', 'religion', 'games_gaming',
       'identity_discrimination_subculture', 'travel_mobility_transport',
       'cloud_networks_security_privacy', 'arts_crafts_design',
       'hardware_devices_infrastructure', 'animals_nature',
       'social_media_plaforms_communities', 'adult_content',
       'life_relationship', 'books_movies_fandom', 'economy_finance',
       'environment_climate_weather', 'music', 'health_mental_health_body',
       'current_political_events', 'food_drinks', 'ai_silicon_valley'],
      dtype='object')

In [44]:
server_topic_df.to_csv("community_type_251219_reduced_merged.csv", index=False)

### get topics that cover 95 percent of all toots per server

In [228]:
def calculate_95_percent(df, topic_model, threshold=0.95):

    server_topic_weighted = (
        df.groupby(["instance", "topic"])["topic_prob"]
        .sum()
        .reset_index(name="weighted_prob_sum")
    )

    server_topic_weighted["relative_share"] = (
        server_topic_weighted
        .groupby("instance")["weighted_prob_sum"]
        .transform(lambda x: x / x.sum())
    )

    server_topic_weighted = server_topic_weighted.sort_values(
        ["instance", "relative_share"],
        ascending=[True, False]
    )


    rows = []

    for instance, group in server_topic_weighted.groupby("instance"):
        total = 0.0

        for _, row in group.iterrows():
            rows.append(row)
            total += row["relative_share"]

            if total >= threshold:
                break

    top_95_topics_per_server = pd.DataFrame(rows)

    topic_info_df = topic_model.get_topic_info()[["Topic", "CustomName"]]
    topic_info_dict = dict(zip(topic_info_df["Topic"], topic_info_df["CustomName"]))

    top_95_topics_per_server["topic_name"] = (
        top_95_topics_per_server["topic"].map(topic_info_dict)
    )

    return top_95_topics_per_server


In [257]:
top_95_df = calculate_95_percent(df, reduced_outliers_model)

In [258]:
top_95_df["instance"].value_counts()

instance
mas.to                        24
mastodon.world                24
nerdculture.de                24
mastodon.uno                  24
norrebro.space                23
                              ..
a-social.techtransthai.org     1
a.katgirl.club                 1
youth.social                   1
zany.tech                      1
0x50.ch                        1
Name: count, Length: 10137, dtype: int64

In [259]:
top_95_df[top_95_df["instance"] == "mastodon.social"]

,instance,topic,weighted_prob_sum,relative_share,topic_name
78015,mastodon.social,11,72.589332,0.099734,Current Political Events
78004,mastodon.social,0,69.685759,0.095745,"Social Media, Platforms & Online Communities"
78021,mastodon.social,17,47.425030,0.065160,"Cloud, Networks, Security & Privacy"
78017,mastodon.social,13,42.585742,0.058511,"Travel, Mobility & Transport"
78008,mastodon.social,4,37.746453,0.051862,Countries & Local Contexts
78013,mastodon.social,9,37.746453,0.051862,Music
78012,mastodon.social,8,35.810737,0.049202,"Books, Movies & Fandom"
78006,mastodon.social,2,31.939306,0.043883,"Environment, Climate & Weather"
78007,mastodon.social,3,29.035733,0.039894,Software & Programming
78005,mastodon.social,1,25.164302,0.034574,Everyday Life & Relationships


# cluster topics

In [ ]:
topic_cols = [c for c in server_topic_df.columns if c != "instance"]
topic_means = server_topic_df[topic_cols].mean().sort_values(ascending=False)

In [ ]:
topic_means

# topic distribution

In [64]:
topic_cols = server_topic_df.columns.drop("instance")
topic_total = server_topic_df[topic_cols].sum()
topic_mean = server_topic_df[topic_cols].mean()

topic_summary = pd.DataFrame({
    "total_weight": topic_total,
    "mean_share": topic_mean
}).sort_values("total_weight", ascending=False)

print(topic_summary)


                                    total_weight  mean_share
social_media_plaforms_communities    1441.341459    0.142186
cloud_networks_security_privacy      1004.339967    0.099077
hardware_devices_infrastructure       669.419047    0.066037
humor_memes_play                      664.997764    0.065601
software_programming                  486.779358    0.048020
misc                                  466.218110    0.045992
life_relationship                     438.151394    0.043223
arts_crafts_design                    401.440030    0.039601
music                                 354.021364    0.034924
ai_silicon_valley                     344.099407    0.033945
current_political_events              343.625926    0.033898
animals_nature                        330.149145    0.032569
environment_climate_weather           326.478738    0.032207
travel_mobility_transport             297.303834    0.029329
food_drinks                           294.686041    0.029070
countries_local_contexts

In [65]:
topic_groups = {

    # Culture, Media & Entertainment
    "books_movies_fandom": "Culture, Media & Entertainment",
    "music": "Culture, Media & Entertainment",
    "arts_crafts_design": "Culture, Media & Entertainment",
    "sports": "Culture, Media & Entertainment",
    "humor_memes_play": "Culture, Media & Entertainment",
    "animals_nature": "Culture, Media & Entertainment",
    "food_drinks": "Culture, Media & Entertainment",
    "games_gaming": "Culture, Media & Entertainment",

    # Technology
    "cloud_networks_security_privacy": "Technology",
    "hardware_devices_infrastructure": "Technology",
    "software_programming": "Technology",
    "social_media_plaforms_communities": "Technology",

    # Politics, Economy & Power
    "news_politics": "Politics, Economy & Power",
    "current_political_events": "Politics, Economy & Power",
    "ai_silicon_valley": "Politics, Economy & Power",
    "countries_local_contexts": "Politics, Economy & Power",
    "economy_finance": "Politics, Economy & Power",

    # Society, Identity and Everyday Life
    "identity_discrimination_subculture": "Society, Identity and Everyday Life",
    "life_relationship": "Society, Identity and Everyday Life",
    "adult_content": "Society, Identity and Everyday Life",
    "religion": "Society, Identity and Everyday Life",


    # Science
    "environment_climate_weather": "Science",
    "science_research": "Science",
    "health_mental_health_body": "Science",

    # Other
    "travel_mobility_transport": "Other",
    "misc": "Other"
}

In [66]:
topic_summary = topic_summary.reset_index().rename(columns={"index": "Topic"})
topic_summary["Group"] = topic_summary["Topic"].map(topic_groups)

In [67]:
topic_summary[topic_summary["Group"].isna()]

,Topic,total_weight,mean_share,Group


In [68]:
group_summary = (
    topic_summary
        .groupby("Group")["mean_share"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
)

In [69]:
group_summary

,Group,mean_share
0,Technology,0.355320
1,"Culture, Media & Entertainment",0.272375
2,"Politics, Economy & Power",0.135646
3,"Society, Identity and Everyday Life",0.086189
4,Other,0.075320
5,Science,0.075150
